# Virtue Foundation Dataset Exploration (DAIS 2026)

This notebook explores the **Databricks Virtue Foundation Dataset** - a comprehensive healthcare and public health dataset focused on India. The dataset contains three main tables:

1. **facilities** - Healthcare facilities with organizational details, contact information, and operational metrics
2. **india_post_pincode_directory** - Geographic postal code directory for India
3. **nfhs_5_district_health_indicators** - National Family Health Survey (NFHS-5) district-level health indicators

Let's dive into each table to understand the data structure and contents.

## 1. Healthcare Facilities Table

The `facilities` table contains detailed information about healthcare organizations and facilities.

In [0]:
%sql
-- Get total number of facilities
SELECT COUNT(*) as total_facilities
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities

In [0]:
%sql
CREATE OR REPLACE TABLE facilities_unique_facebook_links AS
SELECT
  unique_id,
  facebookLink
FROM facilities_clean_org
WHERE facebookLink IS NOT NULL AND trim(facebookLink) <> ''
GROUP BY unique_id, facebookLink

In [0]:
%sql
select count(distinct(facebookLink)) from facilities_unique_facebook_links

In [0]:
%sql
-- Sample records from facilities table
SELECT 
  unique_id,
  name,
  organization_type,
  address_city,
  address_stateOrRegion,
  address_country,
  yearEstablished,
  capacity,
  numberDoctors,
  latitude,
  longitude
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities
LIMIT 10

In [0]:
%sql


In [0]:
%sql
select distinct(name) from databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities

In [0]:
%sql
select count(*) from databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities where organization_type <> 'facility'

In [0]:
%sql
create or replace table facilities_clean_org
as
(
  select * from databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities where organization_type = 'facility'
)


In [0]:
%sql
Mark rows with low confidence (1 doctor and too many specialties)

In [0]:
%sql
-- Analyze columns for potential data accuracy issues in facilities_clean_org

SELECT
  COUNT(*) AS total_rows,
  COUNT(CASE WHEN recency_of_page_update IS NULL OR trim(recency_of_page_update) = '' THEN 1 END) AS missing_recency,
  COUNT(CASE WHEN TRY_CAST(recency_of_page_update AS DATE) > CURRENT_DATE THEN 1 END) AS future_recency,
  COUNT(CASE WHEN latitude IS NULL OR longitude IS NULL THEN 1 END) AS missing_geocode,
  COUNT(CASE WHEN yearEstablished IS NULL OR TRY_CAST(yearEstablished AS INT) > YEAR(CURRENT_DATE) THEN 1 END) AS invalid_year_established,
  COUNT(CASE WHEN numberDoctors IS NULL OR TRY_CAST(numberDoctors AS INT) < 1 THEN 1 END) AS invalid_doctor_count,
  COUNT(CASE WHEN specialties IS NULL OR trim(specialties) = '' THEN 1 END) AS missing_specialties,
  COUNT(CASE WHEN capacity IS NULL OR TRY_CAST(capacity AS INT) < 1 THEN 1 END) AS invalid_capacity
FROM facilities_clean_org

In [0]:
%sql
-- Examine columns that might contain hospital/facility names
SELECT 
  unique_id,
  name,
  address_line1,
  address_line2,
  address_line3,
  description,
  source_content_id,
  officialWebsite
FROM facilities_clean_org
LIMIT 20

In [0]:
%sql
-- Look for patterns that might indicate doctor names vs hospital names
SELECT 
  name,
  numberDoctors,
  specialties,
  description,
  CASE 
    WHEN name LIKE 'Dr.%' OR name LIKE 'Dr %' THEN 'Likely Doctor'
    WHEN name LIKE '%Hospital%' OR name LIKE '%Medical%' OR name LIKE '%Clinic%' OR name LIKE '%Centre%' OR name LIKE '%Center%' THEN 'Likely Facility'
    ELSE 'Unclear'
  END as name_type
FROM facilities_clean_org
WHERE name LIKE 'Dr.%' OR name LIKE 'Dr %'
LIMIT 50

In [0]:
%sql
-- Categorize and count name types in the name column
SELECT 
  CASE 
    WHEN name LIKE 'Dr.%' OR name LIKE 'Dr %' THEN 'Starts with Dr'
    WHEN name LIKE '%Hospital%' THEN 'Contains Hospital'
    WHEN name LIKE '%Medical College%' OR name LIKE '%Medical Centre%' OR name LIKE '%Medical Center%' THEN 'Medical College/Centre'
    WHEN name LIKE '%Clinic%' THEN 'Contains Clinic'
    WHEN name LIKE '%Institute%' THEN 'Contains Institute'
    WHEN name LIKE '%Healthcare%' OR name LIKE '%Health Care%' THEN 'Contains Healthcare'
    WHEN name LIKE '%Nursing Home%' THEN 'Nursing Home'
    ELSE 'Other'
  END as name_category,
  COUNT(*) as count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM facilities_clean_org
GROUP BY name_category
ORDER BY count DESC

In [0]:
%sql
-- Look at samples from the 'Other' category to identify potential doctor names
SELECT 
  name,
  numberDoctors,
  capacity,
  description
FROM facilities_clean_org
WHERE name NOT LIKE '%Hospital%'
  AND name NOT LIKE 'Dr.%' 
  AND name NOT LIKE 'Dr %'
  AND name NOT LIKE '%Medical College%' 
  AND name NOT LIKE '%Medical Centre%' 
  AND name NOT LIKE '%Medical Center%'
  AND name NOT LIKE '%Clinic%'
  AND name NOT LIKE '%Institute%'
  AND name NOT LIKE '%Healthcare%' 
  AND name NOT LIKE '%Health Care%'
  AND name NOT LIKE '%Nursing Home%'
LIMIT 30

## Columns That Could Contain Hospital/Facility Names

Based on analysis of the `facilities_clean_org` table:

### ✅ **Primary Column:**
* **`name`** - The main identifier containing facility names
  * 46.95% contain "Hospital"
  * 16.45% contain "Clinic"
  * 5.93% start with "Dr" (but are still facilities like "Dr DY Patil Medical College and Hospital")
  * 24.19% are in "Other" category (facilities with non-standard names like "Medanta The Medicity", "HCG Cancer Centre", etc.)
  * **Issues:** Some entries may be doctor-affiliated rather than pure facility names

### 🔍 **Secondary/Supplementary Columns:**
* **`description`** - Contains long-form text that often includes the full/proper facility name and additional context
  * Useful for extracting canonical names
  * Example: "Fortis Memorial Research Institute (FMRI), Gurgaon, is a multi-super specialty..."

* **`officialWebsite`** - Website domains that could be parsed to derive facility names
  * Example: "aravind.org", "fortishealthcare.com"
  * Less reliable as a direct name source

### ❌ **Not Suitable:**
* **`address_line1`, `address_line2`, `address_line3`** - These contain street addresses and location descriptors, NOT facility names
* **`source_content_id`, `source_ids`** - These are UUIDs and identifiers, not names
* **`unique_id`** - Primary key UUID

### 📊 **Recommendation:**
For most use cases, use **`name`** as the primary hospital identifier. If you need to clean or validate names, cross-reference with **`description`** which often contains the full proper name.

In [0]:
%sql
-- Create normalized names for duplicate detection
CREATE OR REPLACE TEMP VIEW facilities_normalized AS
SELECT 
  unique_id,
  name AS original_name,
  -- Normalize: lowercase, remove special chars, extra spaces, common suffixes
  regexp_replace(
    regexp_replace(
      regexp_replace(
        lower(trim(name)),
        '(hospital|clinic|medical center|medical centre|health care|healthcare|nursing home|institute|foundation|pvt ltd|private limited|ltd|llp|incorporated|inc|&|and)', 
        ''
      ),
      '[^a-z0-9 ]', 
      ''
    ),
    '\\s+', 
    ' '
  ) AS normalized_name,
  -- Create soundex for phonetic matching
  soundex(name) AS name_soundex,
  address_city,
  address_stateOrRegion,
  capacity,
  numberDoctors,
  latitude,
  longitude
FROM facilities_clean_org
WHERE name IS NOT NULL AND trim(name) <> ''

In [0]:
%sql
-- Find facilities with identical normalized names (exact duplicates after normalization)
SELECT 
  normalized_name,
  COUNT(*) AS duplicate_count,
  COUNT(DISTINCT address_city) AS distinct_cities,
  COUNT(DISTINCT address_stateOrRegion) AS distinct_states,
  COLLECT_LIST(STRUCT(original_name, address_city, address_stateOrRegion, unique_id)) AS facilities
FROM facilities_normalized
WHERE normalized_name IS NOT NULL AND trim(normalized_name) <> ''
GROUP BY normalized_name
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 50

In [0]:
%sql
-- Find facilities with same soundex (phonetically similar names)
-- Soundex groups names that sound alike but may be spelled differently
SELECT 
  name_soundex,
  COUNT(*) AS phonetic_match_count,
  COUNT(DISTINCT normalized_name) AS distinct_normalized_names,
  COLLECT_LIST(DISTINCT original_name) AS sample_names
FROM facilities_normalized
WHERE name_soundex IS NOT NULL
GROUP BY name_soundex
HAVING COUNT(*) > 1 
  AND COUNT(DISTINCT normalized_name) > 1  -- Only where normalized names differ
ORDER BY phonetic_match_count DESC
LIMIT 30

In [0]:
%sql
-- Find potential duplicates using Levenshtein distance (edit distance)
-- This catches typos, minor spelling variations
-- Limited to same city/state to reduce false positives
WITH cross_match AS (
  SELECT 
    f1.unique_id AS id1,
    f2.unique_id AS id2,
    f1.original_name AS name1,
    f2.original_name AS name2,
    f1.normalized_name AS norm1,
    f2.normalized_name AS norm2,
    levenshtein(f1.normalized_name, f2.normalized_name) AS edit_distance,
    f1.address_city,
    f1.address_stateOrRegion
  FROM facilities_normalized f1
  INNER JOIN facilities_normalized f2
    ON f1.address_city = f2.address_city
    AND f1.address_stateOrRegion = f2.address_stateOrRegion
    AND f1.unique_id < f2.unique_id  -- Avoid duplicate pairs
  WHERE levenshtein(f1.normalized_name, f2.normalized_name) BETWEEN 1 AND 3  -- Very similar names
    AND length(f1.normalized_name) > 5  -- Ignore very short names
)
SELECT 
  name1,
  name2,
  edit_distance,
  address_city,
  address_stateOrRegion,
  id1,
  id2
FROM cross_match
ORDER BY edit_distance, name1
LIMIT 100

In [0]:
%sql
-- Create comprehensive duplicate detection table with all techniques
CREATE TABLE IF NOT EXISTS facility_duplicate_candidates AS
WITH 
-- Exact normalized matches
exact_matches AS (
  SELECT 
    f1.unique_id AS facility_id,
    f1.original_name AS facility_name,
    f2.unique_id AS duplicate_id,
    f2.original_name AS duplicate_name,
    'EXACT_NORMALIZED' AS match_type,
    100 AS confidence_score,
    f1.normalized_name AS match_key,
    f1.address_city,
    f1.address_stateOrRegion,
    f1.latitude,
    f1.longitude,
    f2.latitude AS dup_latitude,
    f2.longitude AS dup_longitude
  FROM facilities_normalized f1
  INNER JOIN facilities_normalized f2
    ON f1.normalized_name = f2.normalized_name
    AND f1.unique_id < f2.unique_id
  WHERE f1.normalized_name IS NOT NULL 
    AND trim(f1.normalized_name) <> ''
),
-- Soundex phonetic matches
soundex_matches AS (
  SELECT 
    f1.unique_id AS facility_id,
    f1.original_name AS facility_name,
    f2.unique_id AS duplicate_id,
    f2.original_name AS duplicate_name,
    'SOUNDEX_PHONETIC' AS match_type,
    80 AS confidence_score,
    f1.name_soundex AS match_key,
    f1.address_city,
    f1.address_stateOrRegion,
    f1.latitude,
    f1.longitude,
    f2.latitude AS dup_latitude,
    f2.longitude AS dup_longitude
  FROM facilities_normalized f1
  INNER JOIN facilities_normalized f2
    ON f1.name_soundex = f2.name_soundex
    AND f1.unique_id < f2.unique_id
    AND f1.normalized_name <> f2.normalized_name  -- Different normalized names
  WHERE f1.name_soundex IS NOT NULL
),
-- Levenshtein fuzzy matches
levenshtein_matches AS (
  SELECT 
    f1.unique_id AS facility_id,
    f1.original_name AS facility_name,
    f2.unique_id AS duplicate_id,
    f2.original_name AS duplicate_name,
    'LEVENSHTEIN_FUZZY' AS match_type,
    CASE 
      WHEN levenshtein(f1.normalized_name, f2.normalized_name) = 1 THEN 90
      WHEN levenshtein(f1.normalized_name, f2.normalized_name) = 2 THEN 70
      ELSE 50
    END AS confidence_score,
    CAST(levenshtein(f1.normalized_name, f2.normalized_name) AS STRING) AS match_key,
    f1.address_city,
    f1.address_stateOrRegion,
    f1.latitude,
    f1.longitude,
    f2.latitude AS dup_latitude,
    f2.longitude AS dup_longitude
  FROM facilities_normalized f1
  INNER JOIN facilities_normalized f2
    ON f1.address_city = f2.address_city
    AND f1.address_stateOrRegion = f2.address_stateOrRegion
    AND f1.unique_id < f2.unique_id
  WHERE levenshtein(f1.normalized_name, f2.normalized_name) BETWEEN 1 AND 3
    AND length(f1.normalized_name) > 5
)
SELECT * FROM exact_matches
UNION ALL
SELECT * FROM soundex_matches
UNION ALL
SELECT * FROM levenshtein_matches

In [0]:
%sql
-- Summary of duplicate candidates by match type and confidence
SELECT 
  match_type,
  COUNT(*) AS total_pairs,
  AVG(confidence_score) AS avg_confidence,
  COUNT(DISTINCT facility_id) AS unique_facilities,
  COUNT(DISTINCT CONCAT(address_city, '|', address_stateOrRegion)) AS distinct_locations
FROM facility_duplicate_candidates
GROUP BY match_type
ORDER BY avg_confidence DESC

In [0]:
%sql
-- View top high-confidence duplicate candidates
SELECT 
  facility_name,
  duplicate_name,
  match_type,
  confidence_score,
  address_city,
  address_stateOrRegion,
  -- Calculate geographic distance if both have coordinates
  CASE 
    WHEN latitude IS NOT NULL AND longitude IS NOT NULL 
         AND dup_latitude IS NOT NULL AND dup_longitude IS NOT NULL
    THEN ROUND(
      ST_Distance(
        ST_Point(longitude, latitude),
        ST_Point(dup_longitude, dup_latitude)
      ) / 1000, 2
    )
    ELSE NULL
  END AS distance_km,
  facility_id,
  duplicate_id
FROM facility_duplicate_candidates
WHERE confidence_score >= 70
ORDER BY confidence_score DESC, facility_name
LIMIT 50

In [0]:
%sql
select * from facilities_clean_org where unique_id in ('d56f67f6-1c19-4a42-b0cc-fd8c716faea3', '1193a73b-6e62-4a77-b95f-3ce9b464f37b')

## Facility Name Duplicate Detection - Results Summary

Using state-of-the-art string matching techniques, we identified **62,496 potential duplicate pairs** across 3 different matching approaches:

### Matching Techniques Applied:

#### 1. **Exact Normalized Matching** (100% Confidence)
* **1,196 duplicate pairs** found
* **655 unique facilities** involved
* **Method**: Normalized names by:
  - Converting to lowercase
  - Removing common suffixes (Hospital, Clinic, Medical Center, Healthcare, Ltd, Inc, etc.)
  - Removing special characters and extra spaces
* **Example duplicates**:
  - "Apollo Hospital" appears in 13 different cities
  - "32 Pearls Dental Clinic" found in multiple locations
  - "ASG Eye Hospital" found across 7+ cities

#### 2. **Phonetic Matching (Soundex)** (80% Confidence)
* **60,939 duplicate pairs** found
* **7,593 unique facilities** involved
* **Method**: Groups names that sound alike but may be spelled differently
* **Example matches**:
  - "Sanjeevani Hospital" ↔ "Sanjivani Hospital" ↔ "Sanjeevini Hospital"
  - "Krishna Hospital" ↔ "Krushna Hospital"
  - "Shree Hospital" ↔ "Sri Hospital"

#### 3. **Fuzzy Matching (Levenshtein Distance)** (50-90% Confidence)
* **361 duplicate pairs** found
* **254 unique facilities** involved in same city/state
* **Method**: Detects typos and minor spelling variations (1-3 character edits)
* **Example matches**:
  - "Nidan Hospital" ↔ "Nidaan Hospital" (1 char difference)
  - "Rajeswari Hospital" ↔ "Rajeshwari Nursing Home" (2 chars)
  - "Balaji Hospital" ↔ "Balajee Hospital" (1 char)

### Key Insights:

* **Geographic Distribution**: Many exact matches occur in different cities/states, suggesting legitimate facilities with the same brand name
* **True Duplicates**: Facilities with:
  - Identical normalized names
  - Same city and state
  - Very close geographic coordinates (< 0.1 km)
  - Are most likely to be true duplicates (data quality issues)

* **Chain Facilities**: Names like "Apollo", "Fortis", "Max", "ASG Eye Hospital" appear multiple times legitimately as part of hospital chains

### Output Table:
All results saved in **`facility_duplicate_candidates`** table with columns:
- `facility_id`, `facility_name`, `duplicate_id`, `duplicate_name`
- `match_type`, `confidence_score`
- `address_city`, `address_stateOrRegion`
- `latitude`, `longitude`, `dup_latitude`, `dup_longitude`
- Geographic distance calculation included for coordinate-based validation

In [0]:
%sql
-- Refined duplicate detection: Only consider duplicates if they're in the SAME location
-- This distinguishes true duplicates from legitimate multi-location facilities

CREATE OR REPLACE TABLE facility_true_duplicates AS
WITH location_duplicates AS (
  SELECT 
    f1.unique_id AS facility_id,
    f1.original_name AS facility_name,
    f2.unique_id AS duplicate_id,
    f2.original_name AS duplicate_name,
    f1.normalized_name,
    f1.address_city,
    f1.address_stateOrRegion,
    f1.latitude,
    f1.longitude,
    f2.latitude AS dup_latitude,
    f2.longitude AS dup_longitude,
    -- Calculate geographic distance when coordinates available
    CASE 
      WHEN f1.latitude IS NOT NULL AND f1.longitude IS NOT NULL 
           AND f2.latitude IS NOT NULL AND f2.longitude IS NOT NULL
      THEN ST_Distance(
        ST_Point(f1.longitude, f1.latitude),
        ST_Point(f2.longitude, f2.latitude)
      ) / 1000  -- Convert to kilometers
      ELSE NULL
    END AS distance_km,
    -- Determine duplicate type
    CASE 
      WHEN f1.latitude IS NOT NULL AND f1.longitude IS NOT NULL 
           AND f2.latitude IS NOT NULL AND f2.longitude IS NOT NULL
           AND ST_Distance(
             ST_Point(f1.longitude, f1.latitude),
             ST_Point(f2.longitude, f2.latitude)
           ) / 1000 < 0.5  -- Within 500 meters
      THEN 'TRUE_DUPLICATE_SAME_LOCATION'
      WHEN f1.address_city = f2.address_city 
           AND f1.address_stateOrRegion = f2.address_stateOrRegion
           AND (f1.latitude IS NULL OR f1.longitude IS NULL 
                OR f2.latitude IS NULL OR f2.longitude IS NULL)
      THEN 'POTENTIAL_DUPLICATE_SAME_CITY_NO_COORDS'
      ELSE 'DIFFERENT_LOCATION'
    END AS duplicate_category
  FROM facilities_normalized f1
  INNER JOIN facilities_normalized f2
    ON f1.normalized_name = f2.normalized_name
    AND f1.unique_id < f2.unique_id
    -- Must be in same city AND state to even be considered
    AND f1.address_city = f2.address_city
    AND f1.address_stateOrRegion = f2.address_stateOrRegion
  WHERE f1.normalized_name IS NOT NULL 
    AND trim(f1.normalized_name) <> ''
)
SELECT * FROM location_duplicates
WHERE duplicate_category IN ('TRUE_DUPLICATE_SAME_LOCATION', 'POTENTIAL_DUPLICATE_SAME_CITY_NO_COORDS')
ORDER BY duplicate_category, normalized_name

In [0]:
%sql
-- Summary of location-based duplicates
SELECT 
  duplicate_category,
  COUNT(*) AS duplicate_pairs,
  COUNT(DISTINCT facility_id) AS unique_facilities,
  COUNT(DISTINCT CONCAT(address_city, '|', address_stateOrRegion)) AS distinct_cities,
  ROUND(AVG(distance_km), 2) AS avg_distance_km,
  ROUND(MIN(distance_km), 2) AS min_distance_km,
  ROUND(MAX(distance_km), 2) AS max_distance_km
FROM facility_true_duplicates
GROUP BY duplicate_category
ORDER BY duplicate_pairs DESC

In [0]:
%sql
-- View actual true duplicate records (same location)
SELECT 
  facility_name,
  duplicate_name,
  address_city,
  address_stateOrRegion,
  ROUND(distance_km, 3) AS distance_km,
  duplicate_category,
  facility_id,
  duplicate_id
FROM facility_true_duplicates
WHERE duplicate_category = 'TRUE_DUPLICATE_SAME_LOCATION'
ORDER BY address_city, facility_name

In [0]:
%sql
-- Identify legitimate multi-location facilities (chains)
-- These have the same normalized name but in DIFFERENT cities or states
WITH facility_locations AS (
  SELECT 
    normalized_name,
    COUNT(DISTINCT unique_id) AS total_facilities,
    COUNT(DISTINCT address_city) AS distinct_cities,
    COUNT(DISTINCT address_stateOrRegion) AS distinct_states,
    COLLECT_SET(STRUCT(original_name, address_city, address_stateOrRegion)) AS locations
  FROM facilities_normalized
  WHERE normalized_name IS NOT NULL AND trim(normalized_name) <> ''
  GROUP BY normalized_name
  HAVING COUNT(DISTINCT unique_id) > 1  -- Multiple facilities
    AND (COUNT(DISTINCT address_city) > 1 OR COUNT(DISTINCT address_stateOrRegion) > 1)  -- Different locations
)
SELECT 
  normalized_name,
  total_facilities,
  distinct_cities,
  distinct_states,
  locations
FROM facility_locations
ORDER BY total_facilities DESC, distinct_cities DESC
LIMIT 30

## 📍 Location-Based Duplicate Detection - Refined Analysis

### Why Location Matters

The initial analysis found **1,196 "exact normalized" duplicate pairs** - but this treated facilities with the same name in different cities as duplicates. A "City Clinic" in Mumbai is completely different from a "City Clinic" in Delhi!

### Refined Approach: Location-First Deduplication

We now **only** consider facilities as potential duplicates if they have:
1. **Same normalized name** (after removing common suffixes like Hospital, Clinic, etc.)
2. **Same city AND state**
3. **Geographic proximity** (within 500 meters if coordinates available)

### Results After Location Filtering:

#### ✅ **True Duplicates: Only 11 pairs found**
* Same name, same city/state, **within 500 meters** of each other
* Average distance: **0.0 km** (essentially same location)
* These are **highly likely data quality issues** - same facility entered twice

**Examples:**
* "Om Dental Clinic" appears twice in Ahmedabad (2 meters apart)
* "Sarla Hospital" appears twice in Mumbai (5 meters apart)
* "Government Medical College, Thiruvananthapuram" appears twice (1 meter apart)
* "Krishna Hospital" vs "Krishna Clinic" in Ahmedabad (0 meters apart - likely same facility with name variation)

#### 🏥 **Chain Facilities: NOT Duplicates**

Facilities with the same name in **different locations** are **legitimate multi-location operations**:

| Facility Name | Locations | Cities | States | Type |
|---------------|-----------|--------|--------|------|
| **Apollo** | 13 | 13 | 11 | Major hospital chain |
| **City Hospital** | 8 | 8 | 8 | Common independent facilities |
| **ESI Hospital** | 8 | 8 | 8 | Government employee hospitals |
| **Krishna Hospital** | 8 | 7 | 7 | Multiple independent facilities |
| **ASG Eye Hospital** | 7 | 7 | 7 | Specialty eye hospital chain |
| **City Dental** | 7 | 7 | 7 | Independent dental clinics |
| **Fortis** | 5 | 5 | 5 | Major hospital chain |
| **Max Lab** | 5 | 5 | 5 | Diagnostic lab chain |

### Key Insights:

✅ **True duplicate rate: < 0.2%** (11 pairs out of 10,000 facilities)

✅ **Most "duplicates" are legitimate**: Chain facilities, government hospital networks, or common facility names in different locations

✅ **Geographic validation is critical**: Without coordinates or location matching, we would incorrectly flag 1,196 pairs as duplicates (118x overcounting!)

### Recommendations:

1. **For true duplicates** (11 pairs): Review and merge records, keeping the most complete data
2. **For chain facilities**: Add a "chain_name" or "parent_organization" field to properly track multi-location operations
3. **For future data collection**: Always capture precise location (lat/long) to enable accurate deduplication

### Output Tables:

* **`facility_true_duplicates`** - Only contains location-verified duplicates (same city/state + close proximity)
* **`facility_duplicate_candidates`** - Original comprehensive analysis (includes all matching types)

In [0]:
%sql
select * from facility_true_duplicates

In [0]:
%sql
CREATE OR REPLACE TABLE facilities_cleaned_org_with_duplicate_link AS
SELECT 
  f.*,
  CASE 
    WHEN td.facility_id IS NOT NULL THEN td.duplicate_id
    ELSE NULL
  END AS possible_duplicate_link
FROM facilities_clean_org f
LEFT JOIN facility_true_duplicates td
  ON f.unique_id = td.facility_id

In [0]:
%sql
select * from facilities_cleaned_org_with_duplicate_link where possible_duplicate_link
 is not null

In [0]:
%sql
-- Compare the impact of location-based filtering
SELECT 
  'Original Approach (Name Only)' AS approach,
  'Exact Normalized Match' AS method,
  1196 AS duplicate_pairs,
  655 AS unique_facilities,
  'Treats same name in different cities as duplicates' AS issue
UNION ALL
SELECT 
  'Location-Based Approach' AS approach,
  'Name + Location + Proximity' AS method,
  11 AS duplicate_pairs,
  11 AS unique_facilities,
  'Only flags same name at same location' AS issue
UNION ALL
SELECT 
  'Reduction' AS approach,
  'Impact of Location Filtering' AS method,
  1196 - 11 AS duplicate_pairs,
  655 - 11 AS unique_facilities,
  'Chain facilities and legitimate multi-location operations' AS issue

In [0]:
%sql
select * from facilities_clean_org where numberDoctors IS NULL


In [0]:
%sql
-- Create a table listing all unique cleaned source types with facility counts
CREATE OR REPLACE TABLE unique_source_types AS
SELECT 
  regexp_replace(trim(source_type), '[^a-zA-Z0-9 ]', '') AS source_type_clean,
  COUNT(DISTINCT unique_id) AS facility_count
FROM facilities_clean_org
LATERAL VIEW explode(split(source_types, ',')) AS source_type
WHERE source_type IS NOT NULL AND trim(source_type) <> ''
GROUP BY regexp_replace(trim(source_type), '[^a-zA-Z0-9 ]', '')

In [0]:
%sql
select * from facilities_clean_org where source_types is null


In [0]:
%sql
SELECT * FROM unique_source_types ORDER BY facility_count DESC

In [0]:
%sql
-- Count the number of facilities that have each source type
SELECT 
  regexp_replace(trim(source_type), '[^a-zA-Z0-9 ]', '') AS source_type_clean,
  COUNT(DISTINCT unique_id) AS facility_count
FROM facilities_clean_org
LATERAL VIEW explode(split(source_types, ',')) AS source_type
WHERE source_type IS NOT NULL AND trim(source_type) <> ''
GROUP BY regexp_replace(trim(source_type), '[^a-zA-Z0-9 ]', '')
ORDER BY facility_count DESC

In [0]:
%sql
-- Bar chart: recency_of_page_update (x axis) vs count of facilities
SELECT recency_of_page_update, COUNT(*) AS facility_count
FROM facilities_clean_org
WHERE recency_of_page_update IS NOT NULL AND trim(recency_of_page_update) <> ''
GROUP BY recency_of_page_update
ORDER BY facility_count DESC

In [0]:
%sql
create or replace table facilities_bad_ids
as
(
select * from databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities where organization_type <> 'facility'
)

In [0]:
%sql
select name, count(*) as cnt from databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities group by 1 order by 2 desc

In [0]:
%sql
-- Create a table listing all unique cleaned specialties
CREATE TABLE virtue_foundation_cleaned_specialties AS
SELECT DISTINCT regexp_replace(trim(specialty), '[^a-zA-Z0-9 ]', '') AS specialty_clean
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities
LATERAL VIEW explode(split(specialties, ',')) AS specialty
WHERE specialty IS NOT NULL AND trim(specialty) <> ''

In [0]:
%sql
CREATE OR REPLACE TABLE bronze.unique_id_specialty_mapping AS
SELECT
  f.unique_id,
  COLLECT_SET(regexp_replace(trim(specialty), '[^a-zA-Z0-9 ]', '')) AS cleaned_specialties
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities f
LATERAL VIEW explode(split(f.specialties, ',')) AS specialty
WHERE f.unique_id IS NOT NULL
  AND specialty IS NOT NULL
  AND trim(specialty) <> ''
GROUP BY f.unique_id

In [0]:
%sql
select * from virtue_foundation_cleaned_specialties

In [0]:
%sql
-- Distribution of organization types
SELECT 
  organization_type,
  COUNT(*) as count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities
WHERE organization_type IS NOT NULL
GROUP BY organization_type
ORDER BY count DESC

In [0]:
%sql
-- Top states/regions by facility count
SELECT 
  address_stateOrRegion,
  COUNT(*) as facility_count,
  COUNT(CASE WHEN latitude IS NOT NULL AND longitude IS NOT NULL THEN 1 END) as geocoded_count
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities
WHERE address_stateOrRegion IS NOT NULL
GROUP BY address_stateOrRegion
ORDER BY facility_count DESC
LIMIT 15

## 2. India Post Pincode Directory

This table contains postal code (pincode) information across India with geographic coordinates.

In [0]:
%sql
-- Get total number of pincode entries
SELECT COUNT(*) as total_pincodes,
       COUNT(DISTINCT pincode) as unique_pincodes,
       COUNT(DISTINCT statename) as unique_states,
       COUNT(DISTINCT district) as unique_districts
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory

In [0]:
%sql
-- Sample pincode records
SELECT 
  pincode,
  officename,
  officetype,
  district,
  statename,
  latitude,
  longitude
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory
LIMIT 10

In [0]:
%sql
-- Pincode distribution by state
SELECT 
  statename,
  COUNT(DISTINCT pincode) as unique_pincodes,
  COUNT(*) as total_offices
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory
GROUP BY statename
ORDER BY unique_pincodes DESC
LIMIT 15

## 3. NFHS-5 District Health Indicators

This table contains National Family Health Survey (NFHS-5) data with comprehensive health indicators at the district level across India.

In [0]:
%sql
-- Get total number of district records
SELECT COUNT(*) as total_districts,
       COUNT(DISTINCT state_ut) as unique_states
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators

In [0]:
%sql
-- Sample health indicator records
SELECT 
  district_name,
  state_ut,
  households_surveyed,
  women_15_49_interviewed,
  institutional_birth_5y_pct,
  women_age_15_49_who_are_literate_pct,
  hh_electricity_pct,
  hh_improved_water_pct
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators
LIMIT 10

In [0]:
%sql
-- State-level health indicator averages
SELECT 
  state_ut,
  COUNT(*) as districts_count,
  ROUND(AVG(women_age_15_49_who_are_literate_pct), 2) as avg_female_literacy,
  ROUND(AVG(institutional_birth_5y_pct), 2) as avg_institutional_births,
  ROUND(AVG(hh_electricity_pct), 2) as avg_electricity_access,
  ROUND(AVG(hh_improved_water_pct), 2) as avg_water_access
FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators
WHERE state_ut IS NOT NULL
GROUP BY state_ut
ORDER BY avg_female_literacy DESC
LIMIT 15

## 4. Cross-Table Analysis

Let's explore potential relationships between the tables by analyzing geographic distributions.

In [0]:
%sql
-- Join facilities with health indicators to see facility distribution vs health outcomes
WITH facility_counts AS (
  SELECT 
    address_stateOrRegion as state,
    COUNT(*) as facility_count
  FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities
  WHERE address_stateOrRegion IS NOT NULL
  GROUP BY address_stateOrRegion
),
health_averages AS (
  SELECT 
    state_ut as state,
    ROUND(AVG(women_age_15_49_who_are_literate_pct), 2) as avg_literacy,
    ROUND(AVG(institutional_birth_5y_pct), 2) as avg_inst_births,
    ROUND(AVG(hh_electricity_pct), 2) as avg_electricity
  FROM databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators
  WHERE state_ut IS NOT NULL
  GROUP BY state_ut
)
SELECT 
  COALESCE(f.state, h.state) as state,
  f.facility_count,
  h.avg_literacy,
  h.avg_inst_births,
  h.avg_electricity
FROM facility_counts f
FULL OUTER JOIN health_averages h ON LOWER(TRIM(f.state)) = LOWER(TRIM(h.state))
ORDER BY f.facility_count DESC NULLS LAST
LIMIT 20

In [0]:
%sql
select * from databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities where unique_id = 'aa4b8ccc-fc61-4681-bce6-6443366c5ed2'

In [0]:
%sql
Weighting old updates less,
mismatch data is auto 0,
recency of activity on faceboook (+),
number of sources,
weighting of source information


In [0]:
%sql
facilities_clean_org

In [0]:
%sql
select * from facilities_cleaned_org_with_duplicate_link where (possible_duplicate_link is not null) or  unique_id in (select distinct(possible_duplicate_link) from facilities_cleaned_org_with_duplicate_link )